## uwtools + ecFlow

The notebook demonstrates the use of `uwtools` to configure and run an ecFlow server, and to load an ecFlow workflow suite into the server for execution. As a demo application, it performs verification of a GFS forecast against a GFS leadtime-zero forecast as a source of truth.

First, get to the correct working directory and activate the virtual environment for the demo:

In [ ]:
cd $BASEDIR
source conda/etc/profile.d/conda.sh
conda activate demo

Look at the `uwtools`-enabled configuration file for the demo. It's not necessary to understand all of this now. The top-level `app` block defines various application-level items that are useful throughout the workflow. The `ecflow` block defines server and suite-definition blocks specific to ecFlow. And the `wxvx` block defines parameters for verification to be performed by [wxvx](https://github.com/NOAA-GSL/wxvx). One thing to note looking at the full config is the use of [Jinja2](https://jinja.palletsprojects.com/en/stable/templates/) expressions to correctly tie together various config blocks.

In [ ]:
cat config.yaml

The `ecflow.server` block provides the bare mininum for running the ecFlow server: A definition of the `ECF_HOME` environment variable, which specifies where the server should run and where, by default, it will look for `.h` include files when creating job scripts from `.ecf` scripts:

In [ ]:
uw config realize -i config.yaml --key-path ecflow.server

In this case, `ecflow.server` refers to `app.basedir` for its value, and `app.basedir` refers to the environment variable `BASEDIR` set by the `start` script.

Start the ecFlow server using the config specified in `config.yaml`. Request a JSON-formatted report of important server environment variables, redirect that to `server.json`, and redirect the rest of the output to `server.log`:

In [ ]:
uw ecflow server -c config.yaml --report >server.json 2>server.log &

Wait until the server has started up and written `server.json`:

In [ ]:
while [[ ! -s server.json ]]; do sleep 1; done

View `server.log` and note that there were no errors, and the the server is using a random available TCP port.

Also note that, since `$HOME/.ecflowrc/ssl` did not already contain the SSL certificate files required for secure communication with the ecFlow server, these were generated.

In [ ]:
cat server.log

At server-startup time, `uw ecflow server` also creates a `.h` include file called `server.h`, defining environment variables that will be needed by `.ecf` scripts:

In [ ]:
cat server.h

The `server.json` file, containing current values for some useful server-related environment variables, was also created:

In [ ]:
jq . server.json

Use `jq` to extract values from this file to use to set some crucial environment variables that will allow the ecFlow client tool, `ecflow_client`, to successfully communicate with the server:

In [ ]:
export ECF_PORT=$(jq -r .ECF_PORT server.json)
export ECF_SSL=$(jq -r .ECF_SSL server.json)

With these environment variables in place, `ecflow_client` can be used to ping the server and verify that it is listening:

In [ ]:
ecflow_client --ping

By default, the server starts halted, so place it into running state with the `restart` command:

In [ ]:
ecflow_client --stats | grep Status
ecflow_client --restart
ecflow_client --stats | grep Status

Now the server is ready to be loaded with a suite definition, provided by the `ecflow.suitedef` block in config.yaml. The `uw ecflow realize` command creates a `suite.def` file in the directory specified by `--output-dir` and, under the directory specified by `--scripts-dir`, any `.ecf` scripts defined by `script` blocks in the suite definition:

In [ ]:
uw ecflow realize -c config.yaml --output-dir=. --scripts-dir=.

Here is the `suite.def`, in ecFlow's custom configuration language. Note that the `fetch_truth` task can proceed immediately, but that `extract_grids` truth requires `fetch_truth` to be complete, as it uses the fetched truth data as an input; and that `verification` requires `extract_grids` to be complete, as it uses the extracted grids in verification:

In [ ]:
cat suite.def

The `.ecf` scripts, which are templates used to create executable job scripts that actually run ecFlow tasks, are created in a directory hierarchy based on the suite name, family names (this demo does not use families), and task names:

In [ ]:
tree vx

The `fetch_truth.ecf` file defines a task that will download truth data to verify a forecast against. It defines some help text between the `%manual` and `%end` tags that can be viewed, for example, in the ecFlow GUI application, `ecflow_ui`. When rendered by the server to a job script, the contents of the `server.h` and `common.h` include files will be inlined where the `%include` statements appear.

In [ ]:
cat vx/fetch_truth.ecf

We already saw the contents of the `server.h` file generated by `uw ecflow server`, above. Here's `common.h`, which defines some functions and traps that allow task jobs to communicate their status to the server:

In [ ]:
cat common.h

Here's the `.ecf` file that runs `wxvx` to extract grids from the forecast and truth datasets to be used in verification:

In [ ]:
cat vx/extract_grids.ecf

And, finally, here's the `.ecf` file that runs `wxvx` to perform verification. Note that this task's job will run on the batch system via Slurm, while the two previous tasks ran locally:

In [ ]:
cat vx/verification.ecf

Load the suite into the server:

In [ ]:
ecflow_client --load=suite.def

Verify that the suite was loaded:

In [ ]:
ecflow_client --get

Begin suite execution:

In [ ]:
ecflow_client --begin=vx